# 05 · Retest reliability of AI-ECG age

How much does AI-ECG age change between two ECGs of the same patient, and what explains the change? For each of the ten models, this
notebook computes

1. the measures of retest reliability in every repeat-ECG set of the test split (Table 2, Fig. 2a, Fig. 3, Supplementary Figs. 2 and 4),
2. the distribution of the within-patient differences (Fig. 2c, Supplementary Fig. 1),
3. the share of the within-patient variance explained by time of day, care setting and physiology (Fig. 2b),
4. the night-minus-daytime difference at equal care setting (Supplementary Fig. 3),
5. the accuracy and the retest variation of the predicted sex (Supplementary Table 3),
6. the pull of the estimates toward the mean age (Supplementary Fig. 5).

| Input | Made in | Used for |
|---|---|---|
| `PREDICTIONS/{key}.parquet` | notebook 04 | AI-ECG age and sex logit of the included ECGs of the test split |
| `MIMIC_RECORDS` | notebook 01 | clock time, recording machine (`cart_id`), heart rate, QTc, PR interval, rhythm |
| `MIMIC_REPORT_FLAGS` | notebook 01 | machine-report statements on artifact or noise, non-standard or additional leads, lead reversal |
| `COHORT`, `ECG_CONTEXT` | notebook 02 | inclusion and split; care setting, age and sex at each ECG |
| `PAIRS_48H`, `PAIRS_1H`, `PAIRS_OUT30`, `PAIRS_NIGHT_DAY` | notebook 02 | repeat-ECG sets of the test split |

The results go to `SUMMARY` (`reliability.csv`, `difference_hist.csv`, `decomposition.csv`, `night_day.csv`, `sex.csv`, `slope.csv`,
`slope_by_age.csv`), from which notebooks 07 and 08 build the tables and figures.

**Each patient counts once.** Patients contribute different numbers of pairs. Every pair is weighted by 1 / (number of pairs of its patient
in the set), and every ECG by 1 / (number of ECGs of its patient).

**Confidence intervals** are 95% percentile intervals from `N_BOOT` patient bootstrap resamples. A resample draws patients with replacement
and keeps all their pairs. Each pair keeps the weight it has in the full set, so a patient drawn twice counts twice; the weights are not
recomputed within a resample.

In [ ]:
%run ./00_setup_and_settings.ipynb

## What to run

The analyses of each model are written to their own files under `SUMMARY/by_model/`. A file that exists is not computed again, so an
interrupted run can be resumed; delete the files of a model to compute it again. The tables in `SUMMARY` are assembled from these files
at the end of the notebook, once every model is done.

Run time with the settings of notebook 00 (`N_BOOT` = 1,000): about 2.5–3 minutes per model and 25 minutes for the ten models on a desktop
CPU (the bootstrap of the seven repeat-ECG sets takes about 100 s per model, the decomposition 15 s, the night–daytime comparison 5 s, the
slope 30 s). For a quick functional check, set `RUN_KEYS = ["lima"]` and lower `N_BOOT` in notebook 00 (for example to 50), with `SUMMARY`
pointing to another folder.

In [ ]:
# ============================== WHAT TO RUN ==============================
RUN_KEYS = MODEL_KEYS              # the models to analyse (all ten)
PARTS = SUMMARY / "by_model"       # one file per analysis and model

In [ ]:
import itertools
import math
import time

import pyarrow.parquet as pq
from sklearn.metrics import roc_auc_score

PARTS.mkdir(parents=True, exist_ok=True)
REPEAT_SETS = ["within_48h", "within_1h", "within_1h_same_machine_no_flag", "outpatient_30d",
               "within_48h_ed", "within_48h_ward", "within_48h_icu"]
CARE_SETTINGS = ["ED", "ICU", "intermediate", "outpatient", "observation", "periop", "other", "unknown"]   # indicators; general ward = reference
RHYTHMS = ["af_flutter", "paced", "other"]                                                             # indicators; sinus rhythm = reference
HIST_EDGES = np.arange(-40, 42, 2.0)                 # 2-year bins of the within-patient difference, -40 to 40 years
AGE_EDGES = [18, 25, 30, 35, 40, 45, 50, 55, 60, 65, 70, 75, 80, 85, 89]   # age bins 18-24, 25-29, ..., 80-84, 85-88 years

## ECGs of the test split

One row per included ECG of the test split with the attributes that the analyses need, including the chronological age for the age gap
(AI-ECG age minus chronological age).

In [ ]:
rec = pd.read_parquet(MIMIC_RECORDS, columns=["idx", "subject_id", "minute", "cart_id", "hr", "qtc_ms", "pr_ms", "rhythm"]).set_index("idx")
coh = pd.read_parquet(COHORT, columns=["idx", "include", "split"]).set_index("idx")
ctx = pd.read_parquet(ECG_CONTEXT, columns=["idx", "loc", "age", "sex"]).set_index("idx")
report_flag = pd.read_parquet(MIMIC_REPORT_FLAGS, columns=["idx", "any"]).set_index("idx")["any"]

ecg = rec.loc[coh.index[coh["include"].astype(bool) & (coh["split"] == "test")]].join(ctx)
ecg["flag"] = report_flag.reindex(ecg.index).fillna(False).astype(bool)          # any statement on artifact, leads or lead reversal
ecg["male"] = (ecg["sex"] == "M").astype(float).where(ecg["sex"].notna())
angle = 2 * np.pi * ecg["minute"].astype(float) / 1440                          # clock time as an angle (minute of the day)
ecg["cos"], ecg["sin"] = np.cos(angle), np.sin(angle)
print(f"test split: {masked(len(ecg))} ECGs of {masked(ecg['subject_id'].nunique())} patients")

## Repeat-ECG sets

The three sets of notebook 02 hold ordered pairs of ECGs of the same patient in the test split (`a` the earlier, `b` the later ECG):

- **within 48 h**: every pair recorded more than 0 and at most 48 hours apart (the main set);
- **within 1 h**: pairs of the 48-h set at most 1 hour apart, in the same care setting and during the same admission or emergency
  department stay, or both as outpatients;
- **outpatient within 30 days**: pairs of outpatient ECGs recorded 08:00–17:59 and at most 30 days apart.

Four restricted sets test whether the variation is limited to technically poor recordings or to one care setting: the 1-h pairs recorded on
the same machine (same `cart_id`) whose machine reports carry none of the statements on artifact or noise, non-standard or additional leads,
or lead reversal; and the 48-h pairs with both ECGs in the emergency department, on a general ward or in an intensive care unit.
Each set has its own pair weights, 1 / (number of pairs of the patient in that set).

In [ ]:
ECG_COLS = ["loc", "cart_id", "flag", "cos", "sin", "hr", "qtc_ms", "pr_ms", "rhythm", "age", "male"]


def pair_frame(pairs):
    """The pairs of a set with the attributes of both ECGs (suffix _a: earlier ECG, _b: later ECG)."""
    p = pairs[["idx_a", "idx_b", "subject_id", "hours_apart"]].reset_index(drop=True)
    for s in ("a", "b"):
        assert p[f"idx_{s}"].isin(ecg.index).all(), "a pair contains an ECG outside the included ECGs of the test split"
        e = ecg.loc[p[f"idx_{s}"]]
        for c in ECG_COLS:
            p[f"{c}_{s}"] = e[c].to_numpy()
    return p


def weighted(p):
    """The pairs with the weight 1 / (number of pairs of the patient in this set)."""
    p = p.reset_index(drop=True)
    p["w"] = patient_weights(p["subject_id"])
    return p


p48, p1 = pair_frame(pd.read_parquet(PAIRS_48H)), pair_frame(pd.read_parquet(PAIRS_1H))
same_machine_no_flag = (p1["cart_id_a"] == p1["cart_id_b"]) & ~p1["flag_a"].astype(bool) & ~p1["flag_b"].astype(bool)


def both_in(setting):
    return (p48["loc_a"] == setting) & (p48["loc_b"] == setting)


REPEAT = {"within_48h": weighted(p48),
          "within_1h": weighted(p1),
          "within_1h_same_machine_no_flag": weighted(p1[same_machine_no_flag]),
          "outpatient_30d": weighted(pair_frame(pd.read_parquet(PAIRS_OUT30))),
          "within_48h_ed": weighted(p48[both_in("ED")]),
          "within_48h_ward": weighted(p48[both_in("ward")]),
          "within_48h_icu": weighted(p48[both_in("ICU")])}
assert list(REPEAT) == REPEAT_SETS
display(pd.DataFrame({name: {"patients": masked(p["subject_id"].nunique()), "pairs": masked(len(p))} for name, p in REPEAT.items()}).T)


def with_predictions(p, pred_age):
    """Add the AI-ECG ages of both ECGs, d = later minus earlier AI-ECG age, and the age gaps."""
    q = p.copy()
    q["pred_a"] = pred_age.reindex(q["idx_a"]).to_numpy()
    q["pred_b"] = pred_age.reindex(q["idx_b"]).to_numpy()
    assert q[["pred_a", "pred_b"]].notna().all().all(), "an ECG of a repeat set has no AI-ECG age"
    q["d"] = q["pred_b"] - q["pred_a"]                         # float32, the precision of the predictions written by notebook 04
    for s in ("a", "b"):
        q[f"gap_{s}"] = q[f"pred_{s}"].astype(float) - q[f"age_{s}"]
    return q

## Measures of retest reliability

For each pair, d is the AI-ECG age of the later minus that of the earlier ECG. With the pair weights w:

- **median |d|**: the weighted median of |d|;
- **SD of d** and the **standard error of measurement**, SEM = SD(d) / √2 (the SD of a single ECG around the patient's AI-ECG age without
  retest variation, because d is the difference of two such deviations);
- the **smallest detectable change**, SDC = 1.96 × SD(d) = 2.77 × SEM, and the **95% range** of a single ECG, ±1.96 × SEM;
- the shares of pairs with |d| ≥ 5 and ≥ 10 years;
- the **ICC of the age gap**: one-way random-effects intraclass correlation for single measurements, ICC(1,1), computed from the pairs as
  (MSB − MSW) / (MSB + MSW), with MSB = 2 × the weighted variance of the pair means and MSW = half the weighted mean of the squared
  within-pair differences. It is the share of the variance of the age gap that lies between patients;
- the **ICC of the age gap adjusted for age and sex**: estimates are pulled toward the mean age, so the age gap partly reflects chronological
  age. The gaps of both ECGs of every pair (each with the pair weight) are regressed on age and sex by weighted least squares, and the
  ICC(1,1) is computed from the residuals, in the same pairs and with the same weights; the regression is refitted in every resample;
- the **between-patient SD** of the age gap without retest variation, SEM × √(ICC / (1 − ICC)).

CIs are computed for the median |d| and the two ICCs (one bootstrap loop, the same resamples for the three).
The median interval between the two ECGs is weighted in the same way.

In [ ]:
def icc_1_1(xa, xb, w):
    """One-way random-effects ICC for single measurements from pairs (k = 2), each pair with weight w."""
    m = (xa + xb) / 2
    mu = np.average(m, weights=w)
    msb = 2 * np.average((m - mu) ** 2, weights=w)
    msw = np.average((xa - xb) ** 2, weights=w) / 2
    return float((msb - msw) / (msb + msw))


def icc_adjusted(ga, gb, aa, ab, ma, mb, w):
    """ICC(1,1) of the residuals of the age gap after weighted least-squares regression on age and sex (both ECGs of each pair)."""
    n = len(w)
    X = np.column_stack([np.ones(2 * n), np.r_[aa, ab], np.r_[ma, mb]])
    y, sw = np.r_[ga, gb], np.sqrt(np.r_[w, w])
    beta = np.linalg.lstsq(X * sw[:, None], y * sw, rcond=None)[0]
    e = y - X @ beta
    return icc_1_1(e[:n], e[n:], w)


GAP_COLS = ["gap_a", "gap_b", "age_a", "age_b", "male_a", "male_b"]


def reliability(p, rng):
    """Reliability measures of one set (p: pairs with predictions), with patient bootstrap CIs."""
    w, d = p["w"].to_numpy(), p["d"].to_numpy()
    g = {c: p[c].to_numpy(float) for c in GAP_COLS}
    has_sex = np.isfinite(g["male_a"]) & np.isfinite(g["male_b"])

    def measures(r):
        """Median |d|, ICC of the age gap and adjusted ICC in the rows r (all rows, or one resample)."""
        ra = r[has_sex[r]]
        return (weighted_median(np.abs(d[r]), w[r]),
                icc_1_1(g["gap_a"][r], g["gap_b"][r], w[r]),
                icc_adjusted(*(g[c][ra] for c in GAP_COLS), w[ra]))

    median_abs_d, icc, icc_adj = measures(np.arange(len(p)))
    boot = np.array([measures(r) for r in bootstrap_patients(p["subject_id"], N_BOOT, rng)])
    sd = float(np.sqrt(np.average((d - np.average(d, weights=w)) ** 2, weights=w)))
    sem = sd / math.sqrt(2)
    out = {"patients": int(p["subject_id"].nunique()), "pairs": int(len(p)),
           "median_hours": weighted_median(p["hours_apart"].to_numpy(), w)}
    for j, (name, value) in enumerate((("median_abs_d", median_abs_d), ("icc_gap", icc), ("icc_gap_adj", icc_adj))):
        out[name] = value
        out[f"{name}_lo"], out[f"{name}_hi"] = percentile_ci(boot[:, j])
    out.update(sd_d=sd, sem=sem, sdc=1.96 * sd, range95=1.96 * sem,
               share_ge5_pct=100 * float(np.average(np.abs(d) >= 5, weights=w)),
               share_ge10_pct=100 * float(np.average(np.abs(d) >= 10, weights=w)),
               between_sd=sem * math.sqrt(icc / (1 - icc)))
    return out


RELIABILITY_COLUMNS = ["patients", "pairs", "median_hours", "median_abs_d", "median_abs_d_lo", "median_abs_d_hi", "sd_d", "sem", "sdc",
                       "range95", "share_ge5_pct", "share_ge10_pct", "icc_gap", "icc_gap_lo", "icc_gap_hi", "icc_gap_adj", "icc_gap_adj_lo",
                       "icc_gap_adj_hi", "between_sd"]

## Distribution of the differences

Patient-weighted share of the pairs (in percent) in 2-year bins of d from −40 to 40 years; differences beyond ±40 years are counted in the
outermost bins. A bin that holds 1–9 patients is not shown (`shown` False, no percentage), so that no small count can be read from it.

In [ ]:
def difference_histogram(p):
    d, w = p["d"].to_numpy().astype(float), p["w"].to_numpy()
    b = np.clip(np.digitize(d, HIST_EDGES) - 1, 0, len(HIST_EDGES) - 2)       # bin of each pair; beyond the range -> outermost bin
    subj = p["subject_id"].to_numpy()
    rows = []
    for k in range(len(HIST_EDGES) - 1):
        m = b == k
        n_pat = len(np.unique(subj[m]))
        shown = not (0 < n_pat < SMALL_CELL)
        rows.append({"bin_lo": HIST_EDGES[k], "bin_hi": HIST_EDGES[k + 1],
                     "pct": 100 * w[m].sum() / w.sum() if shown else np.nan, "shown": shown})
    return rows

## Sources of within-patient variation

In the 48-hour set, d is regressed, with the pair weights, on the differences between the two ECGs (later minus earlier) in

- **time of day**: cosine and sine of the clock angle;
- **care setting**: indicators of the care settings (general ward = reference);
- **physiology**: heart rate, QTc, PR interval and indicators of the rhythm (sinus rhythm = reference),

in the pairs with heart rate, QTc and PR interval on both ECGs; the weights are recomputed among these pairs. Indicators that never differ
within a pair are dropped. The R² of the regression on all three groups is divided among the groups by the Shapley decomposition: the
share of a group is its gain in R² when it is added to the other groups, averaged over all orders of adding the groups. The three
shares add up to the R² of the full regression (`all`).

The R² of a weighted least-squares fit depends on the data only through the weighted sums of squares and cross-products of the
regressors and of d. These sums are computed once per patient, and a bootstrap resample adds them with the number of times each patient
was drawn. This gives the same R² as repeating the pairs of each patient as often as the patient was drawn, without refitting the 7
regressions (one for each combination of groups) to the rows of each resample.

In [ ]:
def subset_r2(G, cols, iy):
    """R^2 of the weighted regression of column iy on the intercept (column 0) and the columns cols, from the cross-products G."""
    S = [0] + cols
    beta = np.linalg.lstsq(G[np.ix_(S, S)], G[S, iy], rcond=None)[0]       # normal equations (least squares, in case of a constant column)
    rss = G[iy, iy] - beta @ G[S, iy]
    tss = G[iy, iy] - G[0, iy] ** 2 / G[0, 0]
    return 1 - rss / tss


def shapley_r2(G, cols, iy):
    """Shapley shares of R^2 over the groups of regressors (cols: group -> column numbers), and the R^2 of all groups."""
    names = list(cols)
    n = len(names)
    r2 = {frozenset(): 0.0}
    for k in range(1, n + 1):
        for S in itertools.combinations(names, k):
            r2[frozenset(S)] = subset_r2(G, sum((cols[g] for g in S), []), iy)
    share = {}
    for g in names:
        others = [x for x in names if x != g]
        share[g] = float(sum(math.factorial(len(S)) * math.factorial(n - len(S) - 1) / math.factorial(n)
                             * (r2[frozenset(S) | {g}] - r2[frozenset(S)])
                             for k in range(n) for S in itertools.combinations(others, k)))
    return share, float(r2[frozenset(names)])


def decomposition(p, rng):
    q = p.dropna(subset=[f"{m}_{s}" for m in ("hr", "qtc_ms", "pr_ms") for s in ("a", "b")]).reset_index(drop=True)
    w = patient_weights(q["subject_id"])

    def diff(c):
        return q[f"{c}_b"].to_numpy(float) - q[f"{c}_a"].to_numpy(float)

    def indicator(c, value):
        return (q[f"{c}_b"] == value).to_numpy(float) - (q[f"{c}_a"] == value).to_numpy(float)

    groups = {"time_of_day": np.column_stack([diff("cos"), diff("sin")]),
              "care_setting": np.column_stack([indicator("loc", v) for v in CARE_SETTINGS]),
              "physiology": np.column_stack([diff("hr"), diff("qtc_ms"), diff("pr_ms")] + [indicator("rhythm", v) for v in RHYTHMS])}
    groups = {name: X[:, np.abs(X).sum(0) > 0] for name, X in groups.items()}
    # columns: intercept, the regressors of the three groups, d (last); each scaled to a weighted root mean square of 1, which leaves
    # every R^2 unchanged and keeps the cross-products well conditioned
    A = np.column_stack([np.ones(len(q))] + list(groups.values()) + [q["d"].to_numpy(float)])
    A = A / np.sqrt(np.average(A ** 2, axis=0, weights=w))
    cols, start = {}, 1
    for name, X in groups.items():
        cols[name] = list(range(start, start + X.shape[1]))
        start += X.shape[1]
    iy, m = A.shape[1] - 1, A.shape[1]
    codes, patients = pd.factorize(q["subject_id"])
    per_patient = np.empty((len(patients), m, m))                   # weighted cross-products of each patient's pairs
    for i in range(m):
        for j in range(i, m):
            per_patient[:, i, j] = per_patient[:, j, i] = np.bincount(codes, weights=w * A[:, i] * A[:, j], minlength=len(patients))
    share, total = shapley_r2(per_patient.sum(0), cols, iy)
    boot = []
    for r in bootstrap_patients(patients, N_BOOT, rng):            # r: the patients drawn
        s, t = shapley_r2(np.tensordot(np.bincount(r, minlength=len(patients)), per_patient, axes=1), cols, iy)
        boot.append([s[name] for name in cols] + [t])
    boot = 100 * np.array(boot)
    rows = []
    for j, (name, value) in enumerate(list(share.items()) + [("all", total)]):
        lo, hi = percentile_ci(boot[:, j])
        rows.append({"component": name, "share_pct": 100 * value, "share_pct_lo": lo, "share_pct_hi": hi,
                     "patients": int(q["subject_id"].nunique()), "pairs": int(len(q))})
    return rows

## Night–daytime comparison

Pairs of ECGs of the same patient within 48 hours, one at night (00:00–05:59, `idx_a`) and one in the daytime (10:00–15:59, `idx_b`)
(`PAIRS_NIGHT_DAY`, notebook 02). The within-pair difference (night minus daytime AI-ECG age) is regressed, with the pair weights, on the
differences (night minus daytime ECG) in the care-setting indicators; the intercept is the night-minus-daytime difference at equal care
setting. Indicators that never differ within a pair are dropped, in the full set and in every resample.

In [ ]:
nd = pd.read_parquet(PAIRS_NIGHT_DAY)
nd["w"] = patient_weights(nd["subject_id"])
nd["idx_night"], nd["idx_day"] = nd["idx_a"], nd["idx_b"]
nd["loc_night"] = ecg["loc"].reindex(nd["idx_night"]).to_numpy()
nd["loc_day"] = ecg["loc"].reindex(nd["idx_day"]).to_numpy()
print(f"night-daytime pairs: {masked(nd['subject_id'].nunique())} patients, {masked(len(nd))} pairs")


def night_day(pred_age, rng):
    delta = pred_age.reindex(nd["idx_night"]).to_numpy() - pred_age.reindex(nd["idx_day"]).to_numpy()
    assert np.isfinite(delta).all(), "an ECG of a night-daytime pair has no AI-ECG age"
    w, ln, ld = nd["w"].to_numpy(), nd["loc_night"].to_numpy(), nd["loc_day"].to_numpy()

    def intercept(r):
        X = np.column_stack([(ln[r] == v).astype(float) - (ld[r] == v).astype(float) for v in CARE_SETTINGS])
        Xc = np.column_stack([np.ones(len(r)), X[:, np.abs(X).sum(0) > 0]])
        sw = np.sqrt(w[r])
        return float(np.linalg.lstsq(Xc * sw[:, None], delta[r] * sw, rcond=None)[0][0])

    est = intercept(np.arange(len(nd)))
    lo, hi = percentile_ci([intercept(r) for r in bootstrap_patients(nd["subject_id"], N_BOOT, rng)])
    return [{"estimate": est, "estimate_lo": lo, "estimate_hi": hi, "patients": int(nd["subject_id"].nunique()), "pairs": int(len(nd))}]

## Predicted sex

Nine models also output the logit of male sex (the Lima et al. model has none). The AUC for sex is computed over the ECGs of the test
split with known sex, each ECG weighted by 1 / (number of ECGs of its patient). The predicted sex is the sign of the logit; we report
the weighted share of pairs in which it differs between the two ECGs, within 1 hour and within 48 hours.

In [ ]:
def sex_output(sex_logit):
    e = ecg[ecg["sex"].notna()]
    s = sex_logit.reindex(e.index)
    ok = s.notna().to_numpy()
    auc = roc_auc_score((e["sex"] == "M").to_numpy()[ok], s.to_numpy()[ok], sample_weight=patient_weights(e["subject_id"].to_numpy()[ok]))
    out = {"auc": float(auc)}
    for name, col in (("within_1h", "flip_within_1h_pct"), ("within_48h", "flip_within_48h_pct")):
        p = REPEAT[name]
        sa, sb = sex_logit.reindex(p["idx_a"]).to_numpy(), sex_logit.reindex(p["idx_b"]).to_numpy()
        out[col] = 100 * float(np.average(np.sign(sa) != np.sign(sb), weights=p["w"]))
    return [out]

## Pull toward the mean age

Over the included ECGs of the test split, AI-ECG age is regressed on chronological age by weighted least squares, each ECG with
weight 1 / (number of ECGs of its patient). A slope of 1 means no pull; a smaller slope means that younger patients receive older estimates
and older patients younger ones (the slope of the age gap on age is the slope minus 1). The bootstrap uses per-patient sums of the
weighted cross-products: a resample adds them with the number of times each patient was drawn. Means by 5-year age bin are weighted in the
same way (a bin with 1–9 patients is left empty).

In [ ]:
def slope_fit(S):
    """Slope and intercept of the weighted regression from the sums W, Sx, Sy, Sxx, Sxy."""
    W, Sx, Sy, Sxx, Sxy = S
    b = (W * Sxy - Sx * Sy) / (W * Sxx - Sx ** 2)
    return b, (Sy - b * Sx) / W


def slope_data(pred_age):
    y = pred_age.reindex(ecg.index).to_numpy(float)
    ok = np.isfinite(y)
    return ecg["age"].to_numpy(float)[ok], y[ok], ecg["subject_id"].to_numpy()[ok]


def slope(pred_age, rng):
    x, y, subj = slope_data(pred_age)
    codes, patients = pd.factorize(subj)
    w = 1.0 / np.bincount(codes)[codes]
    per = np.vstack([np.bincount(codes, weights=v, minlength=len(patients)) for v in (w, w * x, w * y, w * x * x, w * x * y)])
    b, _ = slope_fit(per.sum(1))
    boot = [slope_fit(per @ np.bincount(r, minlength=len(patients)))[0] for r in bootstrap_patients(patients, N_BOOT, rng)]
    lo, hi = percentile_ci(boot)
    return [{"slope": float(b), "slope_lo": lo, "slope_hi": hi}]


def slope_by_age(pred_age):
    x, y, subj = slope_data(pred_age)
    w = patient_weights(subj)
    k = np.digitize(x, AGE_EDGES[1:-1])
    rows = []
    for j, (lo, hi) in enumerate(zip(AGE_EDGES[:-1], AGE_EDGES[1:])):
        m = k == j
        n_pat = len(np.unique(subj[m]))
        ok = n_pat >= SMALL_CELL
        rows.append({"age_bin": f"{lo}–{hi - 1}", "lo": lo, "hi": hi - 1, "patients": n_pat if ok else np.nan,
                     "mean_age": float(np.average(x[m], weights=w[m])) if ok else np.nan,
                     "mean_ai_ecg_age": float(np.average(y[m], weights=w[m])) if ok else np.nan,
                     "mean_gap": float(np.average(y[m] - x[m], weights=w[m])) if ok else np.nan})
    return rows

## Run

Each analysis of each model has its own random generator, derived from `BOOT_SEED` and the name of the analysis (`rng_for`), so that the
results of one model do not depend on which other models were run.

In [ ]:
def run_part(name, key, compute):
    """Compute one analysis of one model and write it to PARTS, unless its file exists."""
    f = PARTS / f"{name}_{key}.csv"
    if f.exists():
        print(f"  {name}: exists, kept")
        return
    t0 = time.time()
    df = pd.DataFrame(compute())
    df.insert(0, "key", key)
    df.to_csv(f, index=False)
    print(f"  {name}: {time.time() - t0:.0f} s")


for key in RUN_KEYS:                       # about 2.5-3 minutes per model with N_BOOT = 1000
    print(f"{key} ({model_label(key)})")
    t_model = time.time()
    pred = load_predictions(key).set_index("idx")
    pred_age = pred["pred_age"]
    sets = {name: with_predictions(p, pred_age) for name, p in REPEAT.items()}
    run_part("reliability", key, lambda: [{"set": name, **reliability(p, rng_for(f"reliability {key} {name}"))} for name, p in sets.items()])
    run_part("difference_hist", key, lambda: [{"set": name, **row} for name in ("within_1h", "within_48h")
                                              for row in difference_histogram(sets[name])])
    run_part("decomposition", key, lambda: decomposition(sets["within_48h"], rng_for(f"decomposition {key}")))
    run_part("night_day", key, lambda: night_day(pred_age, rng_for(f"night-daytime {key}")))
    if "sex_logit" in pred.columns:
        run_part("sex", key, lambda: sex_output(pred["sex_logit"]))
    run_part("slope", key, lambda: slope(pred_age, rng_for(f"slope {key}")))
    run_part("slope_by_age", key, lambda: slope_by_age(pred_age))
    print(f"  {key} done in {(time.time() - t_model) / 60:.1f} min")

## Results

The tables in `SUMMARY` are assembled from the files of the ten models, in the order of `MODEL_KEYS`. A table that exists already is kept;
delete it to assemble it again.

In [ ]:
SUMMARY_COLUMNS = {
    "reliability": ["key", "set"] + RELIABILITY_COLUMNS,
    "difference_hist": ["key", "set", "bin_lo", "bin_hi", "pct", "shown"],
    "decomposition": ["key", "component", "share_pct", "share_pct_lo", "share_pct_hi", "patients", "pairs"],
    "night_day": ["key", "estimate", "estimate_lo", "estimate_hi", "patients", "pairs"],
    "sex": ["key", "auc", "flip_within_1h_pct", "flip_within_48h_pct"],
    "slope": ["key", "slope", "slope_lo", "slope_hi"],
    "slope_by_age": ["key", "age_bin", "lo", "hi", "patients", "mean_age", "mean_ai_ecg_age", "mean_gap"],
}


def has_sex_output(key):
    return "sex_logit" in pq.read_schema(PREDICTIONS / f"{key}.parquet").names


def assemble(name):
    """Join the per-model files of one analysis into SUMMARY/<name>.csv once every model is done."""
    keys = [k for k in MODEL_KEYS if name != "sex" or has_sex_output(k)]
    missing = [k for k in keys if not (PARTS / f"{name}_{k}.csv").exists()]
    if missing:
        print(f"{name}.csv: not assembled yet, missing {missing}")
        return None
    df = pd.concat([pd.read_csv(PARTS / f"{name}_{k}.csv") for k in keys], ignore_index=True)[SUMMARY_COLUMNS[name]]
    out = SUMMARY / f"{name}.csv"
    if out.exists():
        print(f"{name}.csv: exists, kept")
    else:
        df.to_csv(out, index=False)
        print(f"{name}.csv: {len(df)} rows")
    return df


tables = {name: assemble(name) for name in SUMMARY_COLUMNS}

In [ ]:
# Overview (aggregate model metrics): SEM within 48 h and within 1 h, ICC of the age gap, adjusted ICC, share explained by all three groups
if all(tables[name] is not None for name in ("reliability", "decomposition", "night_day", "slope")):
    rel = tables["reliability"].set_index(["key", "set"])
    overview = pd.DataFrame({
        "SEM 48 h": rel.xs("within_48h", level="set")["sem"],
        "SEM 1 h": rel.xs("within_1h", level="set")["sem"],
        "ICC gap": rel.xs("within_48h", level="set")["icc_gap"],
        "ICC gap adjusted": rel.xs("within_48h", level="set")["icc_gap_adj"],
        "explained %": tables["decomposition"].set_index(["key", "component"]).xs("all", level="component")["share_pct"],
        "night - day": tables["night_day"].set_index("key")["estimate"],
        "slope": tables["slope"].set_index("key")["slope"]})
    overview.index = [model_label(k) for k in overview.index]
    display(overview.round(3))